# ISRO BAH Hackathon
**Architecture:** CDGM + MRNet (DGMR)  | 

This notebook covers the complete pipeline: environment setup, data loading, model definition, training, and inference.


## 0 · Kaggle Boilerplate


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## 1 · Environment Flags


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["GDAL_PAM_ENABLED"] = "NO"


## 2 · Install Dependencies & Fetch DGMR Code


In [ ]:
# Offline deps + repo (same as your original setup)
!cp -r /kaggle/input/datasets/dakshgoyal123/dgmr-offline-deps/DGMR /kaggle/working/DGMR
!pip install --no-index --find-links /kaggle/input/datasets/dakshgoyal123/dgmr-offline-deps/wheels rasterio natsort timm focal-frequency-loss -q


## 3 · Import CDGM Module


In [ ]:
import sys, shutil

shutil.copy("/kaggle/input/datasets/dakshgoyal123/cdgm-py/cdgm_m3r.py", "/kaggle/working/cdgm_m3r.py")
sys.path.insert(0, "/kaggle/working")

from cdgm_m3r import CDGM
print("CDGM imported successfully")

## 4 · Docstring & Imports
# Imports, logging, and reproducibility helpers.


In [ ]:
"""
finetune_dgmr_lissiv.py
=======================
Finetunes the DGMR MRNet on LISS-IV data.

Key adaptations vs. original M3R code:
  1. Input: 13 channels  (cloudy[3] + sar[2] + temporal[3] + dem[1] +
                           cloud_mask[1] + dist_map[1] + ndvi[1])
     instead of 6 (optical[4] + sar[2]).
  2. Output: 3 channels RGB clear image  (not 4-band planet).
  3. SAR reconstruction head still outputs 2ch (unchanged).
  4. WAB input_resolution changed to (256,256) for 256-px patches.
  5. Partial weight loading: all layers except the first Conv2d (which
     changed from 6->256 to 13->256) and the output convs (4->3).
  6. CDGM optional -- can be disabled via CONFIG to save VRAM/time.
  7. Runs on all available GPUs with DataParallel.

Usage:
    # put DGMR repo code on PYTHONPATH
    PYTHONPATH=/path/to/DGMR/dgmr_code/M3R python finetune_dgmr_lissiv.py
"""

import os, sys, glob, random, copy, logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import lr_scheduler
from collections import OrderedDict

import rasterio
from scipy.ndimage import distance_transform_edt


## 5 · Configuration (CONFIG)
# All tuneable paths, model and training hyper-parameters live here.


In [ ]:
# --------------------------------------------------------------------------- #
#  CONFIG                                                                       #
# --------------------------------------------------------------------------- #
_val_dl_ref = None
CONFIG = {
    # ---- paths ----------------------------------------------------------------
    "data_root":       "/kaggle/input/datasets/aaryamanbisht/satdat/training_data",   # adjust
    "pretrained_mrnet": "/kaggle/input/datasets/daksh387/best-overall/best_overall.pth",         # adjust
    "save_dir":        "/kaggle/working/dgmr_lissiv_ckpt",

    # ---- data -----------------------------------------------------------------
    "patch_size":      256,
    "val_split":       0.1,
    "num_workers":     8,

    # ---- model ----------------------------------------------------------------
    "in_channels":     12,    # your 13-channel input tensor
    "out_channels_opt": 3,    # RGB clear (LISS-IV is 3-band)
    "out_channels_sar": 2,    # SAR reconstruction head (unchanged)
    "feature_sizes":   256,   # K=256  -> DGMR(L)
    "num_arb_layers":  16,    # N=16 ARB layers in CFEM
    "alpha":           0.1,   # residual scaling
    "resolution":      256,   # WAB input_resolution (must match patch_size)
    "window_size":     8,

    # ---- training -------------------------------------------------------------
    "batch_size":      8,
    "epochs":          100,
    "lr":              1e-4,
    "lr_step":         10,
    "lr_gamma":        0.5,
    "use_cdgm":        True,   # set False to skip diffusion guidance
    "cdgm_lr":         2e-5,
    "loss_beta":       0.6,    # SAR loss weight
    "loss_gamma":      0.01,   # NDS loss weight
    "loss_lambda":     0.1,    # CDGM loss weight
    "loss_alpha":      0.5,    # SSIM weight inside Lopt
    "cloud_threshold": 0.10,   # for NDS mask when cloud_mask not in data
    "amp":             True,   # mixed precision
    "seed":            42,
}


## 6 · Logging & Seed
# Deterministic training – same seed gives the same model every run.


In [ ]:
# --------------------------------------------------------------------------- #
#  LOGGING                                                                      #
# --------------------------------------------------------------------------- #
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger(__name__)

# --------------------------------------------------------------------------- #
#  SEED                                                                         #
# --------------------------------------------------------------------------- #
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CONFIG["seed"])


## 7 · Dataset – `SatDataDataset`
# Loads Optical + SAR + Temporal + DEM, computes cloud-mask, NDVI, distance-map on the fly.


In [ ]:
# --------------------------------------------------------------------------- #
#  DATASET  (your existing SatDataDataset, unchanged)                           #
# --------------------------------------------------------------------------- #
class SatDataDataset(Dataset):
    def __init__(self, root_dir, patch_size=256):
        self.root_dir   = root_dir
        self.patch_size = patch_size
        self.files      = glob.glob(os.path.join(root_dir, "cloudy", "*.tif"))
        if not self.files:
            self.files = glob.glob(os.path.join(root_dir, "cloudy", "*.png"))
        if not self.files:
            raise FileNotFoundError(f"No tiles in {os.path.join(root_dir, 'cloudy')}")

    def __len__(self):
        return len(self.files)

    def _load(self, path):
        with rasterio.open(path) as src:
            return src.read().astype(np.float32)

    def __getitem__(self, idx):
        fname    = os.path.basename(self.files[idx])
        base     = self.root_dir

        cloudy   = np.nan_to_num(self._load(os.path.join(base, "cloudy",   fname))[:3], nan=0.0)
        clear    = np.nan_to_num(self._load(os.path.join(base, "clear",    fname))[:3], nan=0.0)
        sar      = np.nan_to_num(self._load(os.path.join(base, "sar",      fname))[:2], nan=0.0)
        temporal = np.nan_to_num(self._load(os.path.join(base, "temporal", fname))[:3], nan=0.0)
        dem      = np.nan_to_num(self._load(os.path.join(base, "dem",      fname))[:1], nan=0.0)

        def n01(x):
            return (x - x.min()) / (x.max() - x.min() + 1e-6)

        cloudy, clear, sar, temporal, dem = map(n01, (cloudy, clear, sar, temporal, dem))

        cloud_mask = (np.abs(cloudy - clear).mean(axis=0) > CONFIG["cloud_threshold"]).astype(np.float32)
        dist_map   = n01(distance_transform_edt(1 - cloud_mask).astype(np.float32))
        ndvi       = n01((temporal[2] - temporal[0]) / (temporal[2] + temporal[0] + 1e-6))

        # 13 channels: cloudy(3) sar(2) temporal(3) dem(1) mask(1) dist(1) ndvi(1)
        inp = np.concatenate([
            cloudy, sar, temporal, dem,
            cloud_mask[np.newaxis], dist_map[np.newaxis], ndvi[np.newaxis],
        ], axis=0).astype(np.float32)

        _, H, W = inp.shape
        ps = self.patch_size
        if H != ps or W != ps:
            sh = (H - ps) // 2
            sw = (W - ps) // 2
            inp   = inp[:,   sh:sh+ps, sw:sw+ps]
            clear = clear[:, sh:sh+ps, sw:sw+ps]

        # augmentation
        if np.random.rand() > 0.5:
            inp   = np.flip(inp,   axis=2).copy()
            clear = np.flip(clear, axis=2).copy()
        if np.random.rand() > 0.5:
            inp   = np.flip(inp,   axis=1).copy()
            clear = np.flip(clear, axis=1).copy()
        k = np.random.randint(0, 4)
        if k:
            inp   = np.rot90(inp,   k, axes=(1, 2)).copy()
            clear = np.rot90(clear, k, axes=(1, 2)).copy()

        return torch.from_numpy(inp), torch.from_numpy(clear.astype(np.float32))


## 8 · Model Components (Swin / WAB / CBAM)
# Window Attention Blocks (WAB) and CBAM attention modules used inside MRNet.


In [ ]:
# --------------------------------------------------------------------------- #
#  MODEL COMPONENTS  (from dgmr_m3r.py, adapted for LISS-IV)                   #
# --------------------------------------------------------------------------- #

# --- Swin/WAB helpers (copied verbatim) ------------------------------------
from timm.models.layers import DropPath, to_2tuple, trunc_normal_

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1  = nn.Linear(in_features, hidden_features)
        self.act  = act_layer()
        self.fc2  = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.drop(self.act(self.fc1(x)))
        return self.drop(self.fc2(x))


def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)


def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)


class WindowAttention(nn.Module):
    def __init__(self, dim, window_size, num_heads, qkv_bias=True,
                 qk_scale=None, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim         = dim
        self.window_size = window_size
        self.num_heads   = num_heads
        head_dim         = dim // num_heads
        self.scale       = qk_scale or head_dim ** -0.5

        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))

        coords_h = torch.arange(window_size[0])
        coords_w = torch.arange(window_size[1])
        coords   = torch.stack(torch.meshgrid([coords_h, coords_w]))
        coords_flatten    = torch.flatten(coords, 1)
        relative_coords   = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords   = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size[0] - 1
        relative_coords[:, :, 1] += window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * window_size[1] - 1
        self.register_buffer("relative_position_index", relative_coords.sum(-1))

        self.qkv      = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj     = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        trunc_normal_(self.relative_position_bias_table, std=.02)
        self.softmax  = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q   = q * self.scale
        attn = q @ k.transpose(-2, -1)
        rpb  = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1],
            self.window_size[0] * self.window_size[1], -1).permute(2, 0, 1).contiguous()
        attn = attn + rpb.unsqueeze(0)
        if mask is not None:
            nW   = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        attn = self.attn_drop(self.softmax(attn))
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        return self.proj_drop(self.proj(x))


class WAB(nn.Module):
    """Window Attention Block -- adapted to accept resolution as constructor arg."""
    def __init__(self, dim, input_resolution, num_heads, window_size=8, shift_size=0,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.dim              = dim
        self.input_resolution = input_resolution
        self.num_heads        = num_heads
        self.window_size      = window_size
        self.shift_size       = shift_size
        if min(input_resolution) <= window_size:
            self.shift_size  = 0
            self.window_size = min(input_resolution)
        assert 0 <= self.shift_size < self.window_size

        self.norm1     = norm_layer(dim)
        self.attn      = WindowAttention(dim, to_2tuple(self.window_size), num_heads,
                                         qkv_bias, qk_scale, attn_drop, drop)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2     = norm_layer(dim)
        self.mlp       = Mlp(dim, int(dim * mlp_ratio), act_layer=act_layer, drop=drop)

        if self.shift_size > 0:
            H, W      = input_resolution
            img_mask  = torch.zeros((1, H, W, 1))
            h_slices  = (slice(0, -self.window_size),
                         slice(-self.window_size, -self.shift_size),
                         slice(-self.shift_size, None))
            w_slices  = (slice(0, -self.window_size),
                         slice(-self.window_size, -self.shift_size),
                         slice(-self.shift_size, None))
            cnt = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = cnt
                    cnt += 1
            mask_windows = window_partition(img_mask, self.window_size).view(-1, self.window_size ** 2)
            attn_mask    = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask    = attn_mask.masked_fill(attn_mask != 0, -100.).masked_fill(attn_mask == 0, 0.)
        else:
            attn_mask = None
        self.register_buffer("attn_mask", attn_mask)

    def forward(self, x):
        B, C, H, W = x.shape
        shortcut   = x
        x          = x.view(B, H, W, C)
        if self.shift_size > 0:
            x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        x_windows  = window_partition(x, self.window_size).view(-1, self.window_size ** 2, C)
        attn_out   = self.attn(x_windows, mask=self.attn_mask)
        attn_out   = attn_out.view(-1, self.window_size, self.window_size, C)
        x          = window_reverse(attn_out, self.window_size, H, W)
        if self.shift_size > 0:
            x = torch.roll(x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        x = x.view(B, C, H, W)
        return shortcut + self.drop_path(x)


# --- Attention blocks (CBAM) -----------------------------------------------
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // 16, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_planes // 16, in_planes, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return self.sigmoid(self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x)))


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv1   = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv1(torch.cat([avg, mx], dim=1)))


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch=256, alpha=0.1):
        super().__init__()
        self.net = nn.Sequential(OrderedDict([
            ("conv1", nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)),
            ("relu1", nn.ReLU(True)),
            ("conv2", nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)),
        ]))
        self.alpha = alpha
        self.ca    = ChannelAttention(out_ch)
        self.sa    = SpatialAttention()

    def forward(self, x):
        out = self.net(x)
        out = self.ca(out) * out
        out = self.sa(out) * out
        return self.alpha * out + x


class ResBlock_A(nn.Module):
    """ARB + WAB variant used in CFEM."""
    def __init__(self, in_ch, out_ch=256, alpha=0.1, resolution=256, window_size=8):
        super().__init__()
        self.net = nn.Sequential(OrderedDict([
            ("conv1", nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)),
            ("relu1", nn.ReLU(True)),
            ("conv2", nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)),
        ]))
        self.tr = WAB(dim=out_ch,
                      input_resolution=[resolution, resolution],
                      num_heads=4,
                      window_size=window_size,
                      shift_size=window_size // 2)

    def forward(self, x):
        return self.tr(self.net(x))


## 9 · MRNet Backbone
# Full multi-modal reconstruction network adapted for LISS-IV 13-channel input → 3-channel RGB output.


In [ ]:
# ---------------------------------------------------------------------------
#  MRNet -- adapted for LISS-IV
#    in_channels : 13  (your 13-ch input)
#    out_opt     : 3   (RGB clear)
#    out_sar     : 2   (SAR reconstruction)
#    The residual addition uses x[:, :3] (cloudy RGB) for optical
#    and x[:, 3:5] (SAR channels) for SAR.
# ---------------------------------------------------------------------------
class MRNet_LISSIV(nn.Module):
    def __init__(self, in_channels=13, out_opt=3, out_sar=2,
                 alpha=0.1, num_layers=16, feature_sizes=256,
                 resolution=256, window_size=8):
        super().__init__()
        R  = resolution
        WS = window_size

        # CFEM: first conv now 13->feature_sizes
        cfem = [
            nn.Conv2d(in_channels, feature_sizes, 3, padding=1, bias=True),
            nn.ReLU(True),
        ]
        for _ in range(num_layers):
            cfem.append(ResBlock(feature_sizes, feature_sizes, alpha))
        self.net = nn.Sequential(*cfem)
        self.csa = ResBlock_A(feature_sizes, feature_sizes, alpha, R, WS)

        # ORM: 3 ARBs + WAB + output conv (-> out_opt channels)
        self.rec_or = nn.Sequential(
            ResBlock(feature_sizes, feature_sizes, alpha),
            ResBlock(feature_sizes, feature_sizes, alpha),
            ResBlock(feature_sizes, feature_sizes, alpha),
        )
        self.fin_org = nn.Conv2d(feature_sizes, out_opt, 3, padding=1, bias=True)
        self.opt_sa  = WAB(dim=out_opt, input_resolution=[R, R],
                           num_heads=1, window_size=WS, shift_size=WS // 2)

        # SRM: 3 ARBs + WAB + output conv (-> out_sar channels)
        self.rec = nn.Sequential(
            ResBlock(feature_sizes, feature_sizes, alpha),
            ResBlock(feature_sizes, feature_sizes, alpha),
            ResBlock(feature_sizes, feature_sizes, alpha),
        )
        self.fin_sr = nn.Conv2d(feature_sizes, out_sar, 3, padding=1, bias=True)
        self.sar_sa = WAB(dim=out_sar, input_resolution=[R, R],
                          num_heads=1, window_size=WS, shift_size=WS // 2)

        # Feature projection for CDGM conditioning
        self.feat   = nn.Conv2d(feature_sizes, out_opt, 3, padding=1, bias=True)

        # store channel slices for residual additions
        self._opt_slice = slice(0, out_opt)       # channels 0:3
        self._sar_slice = slice(out_opt, out_opt + out_sar)  # channels 3:5

    def forward(self, x):
        x_opt = x[:, self._opt_slice]  # cloudy RGB  [B,3,H,W]
        x_sar = x[:, self._sar_slice]  # SAR 2-ch    [B,2,H,W]

        op1 = self.net(x)
        op1 = self.csa(op1) + op1

        # ORM
        op11   = self.rec_or(op1)
        op2    = self.fin_org(op11)
        op2_up = self.opt_sa(op2) + x_opt   # residual to cloudy RGB

        # SRM
        recon     = self.rec(op1)
        recon1    = self.fin_sr(recon)
        recon2_up = self.sar_sa(recon1) + x_sar  # residual to SAR

        # Feature for CDGM
        feat = self.feat(op1)

        return op2_up, recon2_up, feat  # (PCf[B,3], PS[B,2], feat[B,3])


## 10 · Loss Functions
# L_opt (Spectral+Spatial) · L_sar (Masked SAR) · L_chg (NDS consistency) · L_ddpm (Diffusion).


In [ ]:
# --------------------------------------------------------------------------- #
#  LOSS FUNCTIONS                                                               #
# --------------------------------------------------------------------------- #
def gaussian_kernel_loss(size=5, channels=3, sigma=1, dtype=torch.float):
    interval = (2 * sigma + 1) / size
    ax = np.linspace(-(size - 1) / 2., (size - 1) / 2., size)
    xx, yy = np.meshgrid(ax, ax)
    kernel  = np.exp(-0.5 * (np.square(xx) + np.square(yy)) / np.square(sigma))
    kernel /= kernel.sum()
    kt = torch.as_tensor(kernel, dtype=dtype).repeat(channels, 1, 1, 1)
    return kt


def gaussian_conv2d(x, g_kernel):
    ch      = g_kernel.shape[0]
    padding = g_kernel.shape[-1] // 2
    return F.conv2d(x, weight=g_kernel.to(x.device), stride=1, padding=padding, groups=ch)


def create_laplacian_pyramid(x, kernel, levels):
    up   = nn.Upsample(scale_factor=2)
    pyrs = []
    cur  = x
    for _ in range(levels):
        gf   = gaussian_conv2d(cur, kernel)
        down = gf[:, :, ::2, ::2]
        pyrs.append(cur - up(down))
        cur  = down
    pyrs.append(cur)
    return pyrs


class LaplacianPyramidLoss(nn.Module):
    def __init__(self, max_levels=3, channels=3, kernel_size=5, sigma=1):
        super().__init__()
        self.max_levels = max_levels
        self.kernel     = gaussian_kernel_loss(kernel_size, channels, sigma)

    def forward(self, x, target):
        ip = create_laplacian_pyramid(x,      self.kernel, self.max_levels)
        tp = create_laplacian_pyramid(target, self.kernel, self.max_levels)
        return sum(F.l1_loss(a, b) for a, b in zip(ip, tp))


def ssim_loss(pred, target, window_size=11, reduction="mean"):
    """Simple differentiable SSIM loss (1 - SSIM)."""
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ch     = pred.shape[1]
    k      = torch.ones(ch, 1, window_size, window_size, device=pred.device) / (window_size ** 2)
    pad    = window_size // 2

    mu1  = F.conv2d(pred,   k, padding=pad, groups=ch)
    mu2  = F.conv2d(target, k, padding=pad, groups=ch)
    mu1_sq = mu1 ** 2;  mu2_sq = mu2 ** 2;  mu12 = mu1 * mu2
    s1  = F.conv2d(pred   ** 2, k, padding=pad, groups=ch) - mu1_sq
    s2  = F.conv2d(target ** 2, k, padding=pad, groups=ch) - mu2_sq
    s12 = F.conv2d(pred * target, k, padding=pad, groups=ch) - mu12

    ssim = ((2 * mu12 + C1) * (2 * s12 + C2)) / \
           ((mu1_sq + mu2_sq + C1) * (s1 + s2 + C2))
    loss = 1 - ssim
    return loss.mean() if reduction == "mean" else loss


try:
    from focal_frequency_loss import FocalFrequencyLoss as FFL
    _ffl_opt = FFL(loss_weight=1.0, alpha=1.0)
    _ffl_sar = FFL(loss_weight=1.0, alpha=1.0)
    USE_FFL  = True
except ImportError:
    log.warning("focal_frequency_loss not found -- FFL disabled (pip install focal-frequency-loss)")
    USE_FFL  = False


def compute_loss(pred_opt, pred_sar, feat,
                 clear, sar_gt, cloud_mask,
                 cloudy,
                 lap_opt, lap_sar,
                 cfg):
    """
    Compute the full DGMR loss for LISS-IV.

    pred_opt  : [B, 3, H, W]   predicted cloud-free RGB
    pred_sar  : [B, 2, H, W]   predicted SAR
    feat      : [B, 3, H, W]   feature map (for CDGM conditioning)
    clear     : [B, 3, H, W]   ground truth cloud-free
    sar_gt    : [B, 2, H, W]   SAR input (used as SAR GT)
    cloud_mask: [B, 1, H, W]   binary cloud mask  (1 = cloud)
    cloudy    : [B, 3, H, W]   cloudy optical input
    """
    # expand mask to channel dim
    m3  = cloud_mask.expand_as(pred_opt)   # [B,3,H,W]
    m2  = cloud_mask.expand_as(pred_sar)   # [B,2,H,W]

    # --- Lopt -----------------------------------------------------------------
    l_lap_opt = lap_opt(pred_opt, clear)
    l_ssim    = ssim_loss(pred_opt, clear)
    l_ffl_opt = _ffl_opt(pred_opt, clear) if USE_FFL else torch.tensor(0., device=pred_opt.device)
    L_opt     = l_lap_opt + cfg["loss_alpha"] * l_ssim + l_ffl_opt

    # --- Lsar -----------------------------------------------------------------
    l_lap_sar = lap_sar(pred_sar * m2, sar_gt * m2)
    l_ssim_s  = ssim_loss(pred_sar * m2, sar_gt * m2)
    l_ffl_sar = _ffl_sar(pred_sar * m2, sar_gt * m2) if USE_FFL else torch.tensor(0., device=pred_opt.device)
    L_sar     = l_lap_sar + cfg["loss_alpha"] * l_ssim_s + l_ffl_sar

    # --- NDS (Lchg) -----------------------------------------------------------
    nc  = 1 - m3
    d1  = torch.abs(cloudy - clear) * nc
    d2  = torch.abs(cloudy - pred_opt) * nc
    L_chg = F.l1_loss(d1, d2)

    L_primary = L_opt + cfg["loss_beta"] * L_sar + cfg["loss_gamma"] * L_chg
    return L_primary


def normalize_to_01_cw(t):
    mn = t.amin(dim=(2, 3), keepdim=True)
    mx = t.amax(dim=(2, 3), keepdim=True)
    return (t - mn) / (mx - mn).clamp(min=1e-8)


## 11 · Load Pre-trained Weights (Partial)
# Partial weight loading skips the input conv (6→12 ch) and output conv (4→3 ch).


In [ ]:
# --------------------------------------------------------------------------- #
#  LOAD PRETRAINED WEIGHTS (partial)                                            #
# --------------------------------------------------------------------------- #
def load_pretrained_partial(model, ckpt_path):
    """
    Load pretrained M3R weights into MRNet_LISSIV.
    Skips keys with shape mismatch (first conv: 6->256 vs 13->256,
    and output convs: 4->3, 2 stays).
    """
    if not os.path.exists(ckpt_path):
        log.warning(f"Pretrained checkpoint not found at {ckpt_path} -- training from scratch.")
        return

    ckpt = torch.load(ckpt_path, map_location="cpu")
    # handle both raw state_dict and wrapped {'network': ...} format
    sd   = ckpt.get("network", ckpt)

    model_sd   = model.state_dict()
    loaded, skipped = [], []

    for k, v in sd.items():
        if k not in model_sd:
            skipped.append(f"{k} [not in model]")
            continue
        if model_sd[k].shape != v.shape:
            skipped.append(f"{k} [shape {v.shape} vs {model_sd[k].shape}]")
            continue
        model_sd[k] = v
        loaded.append(k)

    model.load_state_dict(model_sd)
    log.info(f"Loaded {len(loaded)} layers from pretrained checkpoint.")
    if skipped:
        log.info(f"Skipped {len(skipped)} layers (shape mismatch / new):")
        for s in skipped[:10]:
            log.info(f"    {s}")
        if len(skipped) > 10:
            log.info(f"    ... and {len(skipped)-10} more")


## 12 · Metrics
# PSNR and SSIM helpers.


In [ ]:
# --------------------------------------------------------------------------- #
#  METRICS                                                                      #
# --------------------------------------------------------------------------- #
def psnr(pred, target, max_val=1.0):
    mse = F.mse_loss(pred, target)
    return 10 * torch.log10(max_val ** 2 / (mse + 1e-8))


## 13 · Training Loop
# Main epoch loop with AMP, LR scheduling, validation, and checkpoint saving.


In [ ]:
# --------------------------------------------------------------------------- #
#  TRAINING LOOP                                                                #
# --------------------------------------------------------------------------- #
def train():
    cfg = CONFIG
    os.makedirs(cfg["save_dir"], exist_ok=True)

    # ---- Dataset -------------------------------------------------------------
    full    = SatDataDataset(cfg["data_root"], cfg["patch_size"])
    val_n   = max(1, int(cfg["val_split"] * len(full)))
    train_n = len(full) - val_n
    train_ds, val_ds = random_split(full, [train_n, val_n])

    train_dl = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,
                          num_workers=cfg["num_workers"], pin_memory=True, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=cfg["batch_size"], shuffle=False,
                      num_workers=cfg["num_workers"], pin_memory=True)
    global _val_dl_ref
    _val_dl_ref = val_dl

    log.info(f"Train: {train_n}  Val: {val_n}  Steps/epoch: {len(train_dl)}")

    # ---- Model ---------------------------------------------------------------
    model = MRNet_LISSIV(
        in_channels   = cfg["in_channels"],
        out_opt       = cfg["out_channels_opt"],
        out_sar       = cfg["out_channels_sar"],
        alpha         = cfg["alpha"],
        num_layers    = cfg["num_arb_layers"],
        feature_sizes = cfg["feature_sizes"],
        resolution    = cfg["resolution"],
        window_size   = cfg["window_size"],
    )
    # load model weights
    try:
        raw_ckpt = torch.load(cfg["pretrained_mrnet"], map_location="cpu")
        if "state_dict" in raw_ckpt:
            model_sd = model.state_dict()
            for k, v in raw_ckpt["state_dict"].items():
                if k in model_sd and model_sd[k].shape == v.shape:
                    model_sd[k] = v
            model.load_state_dict(model_sd)
            log.info(f"Resumed from epoch {raw_ckpt['epoch']} | PSNR {raw_ckpt['val_psnr']:.3f}")
        else:
            raw_ckpt = None
            load_pretrained_partial(model, cfg["pretrained_mrnet"])
    except Exception as e:
        log.warning(f"Could not load checkpoint: {e} -- training from scratch")
        raw_ckpt = None
    # DataParallel across all GPUs
    if torch.cuda.device_count() > 1:
        log.info(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
        model = nn.DataParallel(model)
    model = model.cuda()

    # ---- CDGM (optional) -----------------------------------------------------
    cdgm = None
    opt_cdgm = None
    sched_cdgm = None
    if cfg["use_cdgm"]:
        try:
            sys.path.insert(0, "/kaggle/working")
            from cdgm_m3r import CDGM
            # CDGM(out_opt_ch, in_channels_cond, out_ch, beta_schedule)
            # cond = feat[B,3] + sar_gt[B,2] = 5 channels
            beta_schedule = {
                "train": {"schedule": "sigmoid", "n_timestep": 2000,
                          "linear_start": 1e-6, "linear_end": 0.01},
                "test":  {"schedule": "sigmoid", "n_timestep": 1000,
                          "linear_start": 1e-4, "linear_end": 0.09},
            }
            cdgm = CDGM(
                cfg["out_channels_opt"],   # y_0 channels  (clear RGB = 3)
                cfg["out_channels_opt"] + cfg["out_channels_sar"],  # cond channels (feat3 + sar2 = 5)
                cfg["out_channels_opt"],   # output channels
                beta_schedule,
            )
            cdgm.set_new_noise_schedule(phase="train")
            cdgm.set_loss()
            if torch.cuda.device_count() > 1:
                cdgm = nn.DataParallel(cdgm)
            cdgm = cdgm.cuda()
            opt_cdgm   = torch.optim.Adam(cdgm.parameters(), lr=cfg["cdgm_lr"])
            sched_cdgm = lr_scheduler.StepLR(opt_cdgm, step_size=cfg["lr_step"], gamma=cfg["lr_gamma"])
            log.info("CDGM loaded successfully.")
        except Exception as e:
            log.warning(f"CDGM could not be loaded ({e}). Continuing without diffusion guidance.")
            cdgm = None

    # ---- Optimizer & scheduler -----------------------------------------------
    opt   = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    sched = lr_scheduler.StepLR(opt, step_size=cfg["lr_step"], gamma=cfg["lr_gamma"])
    
    if raw_ckpt is not None and "optimizer" in raw_ckpt:
        try:
            opt.load_state_dict(raw_ckpt["optimizer"])
            log.info("Optimizer state restored")
        except Exception as e:
            log.warning(f"Could not restore optimizer state: {e} -- using fresh optimizer")

    # ---- Loss helpers --------------------------------------------------------
    lap_opt = LaplacianPyramidLoss(channels=cfg["out_channels_opt"]).cuda()
    lap_sar = LaplacianPyramidLoss(channels=cfg["out_channels_sar"]).cuda()

    scaler  = torch.cuda.amp.GradScaler(enabled=cfg["amp"])

    # ---- Training ------------------------------------------------------------
    best_psnr = 0.0

    for epoch in range(18, cfg["epochs"] + 1):
        model.train()
        if cdgm is not None:
            cdgm.train()

        total_loss = 0.0

        for step, (inp, clear) in enumerate(train_dl):
            inp   = inp.cuda(non_blocking=True)    # [B, 13, H, W]
            clear = clear.cuda(non_blocking=True)  # [B, 3,  H, W]

            # split auxiliary channels for SAR GT and cloud mask
            # inp layout: cloudy[0:3] sar[3:5] temporal[5:8] dem[8] mask[9] dist[10] ndvi[11] (total 12 -> check)
            # Your concat order: cloudy(3)+sar(2)+temporal(3)+dem(1)+mask(1)+dist(1)+ndvi(1) = 12... wait
            # Actually: 3+2+3+1+1+1+1 = 12, but CONFIG says in_channels=13.
            # Re-check: cloudy=3, sar=2, temporal=3, dem=1, cloud_mask=1, dist_map=1, ndvi=1 => 12.
            # If your data has a 13th channel, adjust slice below. We use 12 here defensively.
            sar_gt     = inp[:, 3:5]    # SAR 2ch
            cloud_mask = inp[:, 9:10]   # cloud mask 1ch (binary, after sar+temporal+dem)
            cloudy_rgb = inp[:, 0:3]    # cloudy RGB

            opt.zero_grad()
            if opt_cdgm is not None:
                opt_cdgm.zero_grad()

            with torch.cuda.amp.autocast(enabled=cfg["amp"]):
                pred_opt, pred_sar, feat = model(inp)

                L_primary = compute_loss(
                    pred_opt, pred_sar, feat,
                    clear, sar_gt, cloud_mask, cloudy_rgb,
                    lap_opt, lap_sar, cfg,
                )

                L_total = L_primary

                if cdgm is not None:
                    # condition = feat (3ch) + sar_gt (2ch)
                    cond       = normalize_to_01_cw(torch.cat([feat.detach(), sar_gt], dim=1))
                    clear_norm = normalize_to_01_cw(clear)
                    loss_diff  = cdgm(y_0=clear_norm, y_cond=cond)
                    L_total    = L_primary + cfg["loss_lambda"] * loss_diff.mean()

            scaler.scale(L_total).backward()
            scaler.step(opt)
            if opt_cdgm is not None:
                scaler.step(opt_cdgm)
            scaler.update()

            total_loss += L_primary.item()

            if (step + 1) % 50 == 0:
                log.info(f"Epoch {epoch:03d} | Step {step+1:04d}/{len(train_dl)} "
                         f"| Loss {L_primary.item():.4f}")

        sched.step()
        if sched_cdgm is not None:
            sched_cdgm.step()

        avg_loss = total_loss / len(train_dl)
        log.info(f"Epoch {epoch:03d} avg train loss: {avg_loss:.4f}")

        # ---- Validation ------------------------------------------------------
        model.eval()
        val_psnr_sum = 0.0
        with torch.no_grad():
            for inp, clear in val_dl:
                inp   = inp.cuda(non_blocking=True)
                clear = clear.cuda(non_blocking=True)
                with torch.cuda.amp.autocast(enabled=cfg["amp"]):
                    pred_opt, _, _ = model(inp)
                val_psnr_sum += psnr(pred_opt.clamp(0, 1), clear).item()

        avg_val_psnr = val_psnr_sum / len(val_dl)
        log.info(f"Epoch {epoch:03d} val PSNR: {avg_val_psnr:.3f} dB")

        # ---- Save checkpoint -------------------------------------------------
        is_best = avg_val_psnr > best_psnr
        if is_best:
            best_psnr = avg_val_psnr

        raw_model = model.module if isinstance(model, nn.DataParallel) else model
        ckpt = {
            "epoch":       epoch,
            "state_dict":  raw_model.state_dict(),
            "optimizer":   opt.state_dict(),
            "val_psnr":    avg_val_psnr,
            "config":      cfg,
        }
        torch.save(ckpt, os.path.join(cfg["save_dir"], f"epoch_{epoch:03d}.pth"))
        if is_best:
            torch.save(ckpt, os.path.join(cfg["save_dir"], "best.pth"))
            new_file = f"/kaggle/working/best_epoch{epoch:03d}.pth"
            shutil.copy(os.path.join(cfg["save_dir"], "best.pth"), new_file)
            
            # always keep the overall best as a separate file (never deleted)
            shutil.copy(
                os.path.join(cfg["save_dir"], "best.pth"),
                "/kaggle/working/best_overall.pth"
            )
            
            # keep only last 3 recent best checkpoints
            best_files = sorted(glob.glob("/kaggle/working/best_epoch*.pth"))
            for old_file in best_files[:-3]:
                os.remove(old_file)
                log.info(f"  Removed old checkpoint: {old_file}")
            
            log.info(f"  ** New best PSNR: {best_psnr:.3f} dB -- saved best.pth")

    log.info(f"Training complete. Best val PSNR: {best_psnr:.3f} dB")


## 14 · Inference Helper
# Single-sample forward pass utility used during evaluation.


In [ ]:
# --------------------------------------------------------------------------- #
#  INFERENCE HELPER                                                             #
# --------------------------------------------------------------------------- #
def load_for_inference(ckpt_path, cfg=None):
    """Returns a ready-to-use MRNet_LISSIV in eval mode."""
    if cfg is None:
        cfg = CONFIG
    model = MRNet_LISSIV(
        in_channels   = cfg["in_channels"],
        out_opt       = cfg["out_channels_opt"],
        out_sar       = cfg["out_channels_sar"],
        alpha         = cfg["alpha"],
        num_layers    = cfg["num_arb_layers"],
        feature_sizes = cfg["feature_sizes"],
        resolution    = cfg["resolution"],
        window_size   = cfg["window_size"],
    )
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd   = ckpt.get("state_dict", ckpt)
    model.load_state_dict(sd)
    model.eval()
    return model



train()


## 15 · Copy Best Checkpoint


In [ ]:
import shutil
shutil.copy("/kaggle/working/dgmr_lissiv_ckpt/best.pth",
            "/kaggle/working/best_final.pth")

## 16 · Build Validation DataLoader


In [ ]:
full    = SatDataDataset(CONFIG["data_root"], CONFIG["patch_size"])
val_n   = max(1, int(CONFIG["val_split"] * len(full)))
train_n = len(full) - val_n
train_ds, val_ds = random_split(full, [train_n, val_n], 
                                generator=torch.Generator().manual_seed(11))
val_dl  = DataLoader(val_ds, batch_size=4, shuffle=True,
                     num_workers=4, pin_memory=False)
train_dl = DataLoader(train_ds, batch_size=4, shuffle=False,
                       num_workers=4, pin_memory=False)
print(f"Val tiles: {val_n}")

## 17 · Quick Visual Check – Sample Prediction


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from scipy.ndimage import distance_transform_edt

# ---- load best checkpoint ----
model = MRNet_LISSIV(
    in_channels    = CONFIG["in_channels"],
    out_opt        = CONFIG["out_channels_opt"],
    out_sar        = CONFIG["out_channels_sar"],
    alpha          = CONFIG["alpha"],
    num_layers     = CONFIG["num_arb_layers"],
    feature_sizes  = CONFIG["feature_sizes"],
    resolution     = CONFIG["resolution"],
    window_size    = CONFIG["window_size"],
)
ckpt = torch.load("//kaggle/input/datasets/daksh387/best-overall/best_overall.pth", map_location="cpu")
model.load_state_dict(ckpt["state_dict"])
model.eval().cuda()
print(f"Loaded checkpoint from epoch {ckpt['epoch']} | Val PSNR: {ckpt['val_psnr']:.3f} dB")

# ---- grab one batch from val set ----
inp, clear = next(iter(_val_dl_ref))
inp   = inp.cuda()
clear = clear.cuda()

with torch.no_grad():
    pred_opt, _, _ = model(inp)

pred_opt = pred_opt.clamp(0, 1)

# ---- visualize 4 samples ----
fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for i in range(4):
    cloudy_rgb = inp[i, 0:3].cpu().permute(1,2,0).numpy()
    clear_rgb  = clear[i].cpu().permute(1,2,0).numpy()
    pred_rgb   = pred_opt[i].cpu().permute(1,2,0).numpy()

    # per-image psnr
    mse  = ((pred_rgb - clear_rgb) ** 2).mean()
    psnr = 10 * np.log10(1.0 / (mse + 1e-8))

    axes[i,0].imshow(cloudy_rgb.clip(0,1))
    axes[i,0].set_title("Cloudy Input")
    axes[i,0].axis("off")

    axes[i,1].imshow(pred_rgb.clip(0,1))
    axes[i,1].set_title(f"DGMR Output (PSNR={psnr:.2f})")
    axes[i,1].axis("off")

    axes[i,2].imshow(clear_rgb.clip(0,1))
    axes[i,2].set_title("Ground Truth")
    axes[i,2].axis("off")

plt.suptitle(f"DGMR Inference -- Best Checkpoint Epoch {ckpt['epoch']}", 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("/kaggle/working/inference_results.png", dpi=300, bbox_inches='tight')
plt.show()
print("Saved to /kaggle/working/inference_results.png")

## 18 · Batch PSNR / SSIM Evaluation


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

device = next(model.parameters()).device
model.eval()

# ---------------------------------------------------------------- #
# SSIM (needed since it's not defined elsewhere in the notebook)
# ---------------------------------------------------------------- #
def ssim_metric(pred, target, window_size=11):
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ch  = pred.shape[1]
    k   = torch.ones(ch, 1, window_size, window_size, device=pred.device) / (window_size ** 2)
    pad = window_size // 2

    mu1 = F.conv2d(pred,   k, padding=pad, groups=ch)
    mu2 = F.conv2d(target, k, padding=pad, groups=ch)
    mu1_sq, mu2_sq, mu12 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    s1  = F.conv2d(pred   ** 2, k, padding=pad, groups=ch) - mu1_sq
    s2  = F.conv2d(target ** 2, k, padding=pad, groups=ch) - mu2_sq
    s12 = F.conv2d(pred * target, k, padding=pad, groups=ch) - mu12

    ssim_map = ((2 * mu12 + C1) * (2 * s12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (s1 + s2 + C2))
    return ssim_map.mean()

# ---------------------------------------------------------------- #
# scan dataset, track best (lowest) MSE and best (highest) SSIM
# ---------------------------------------------------------------- #
scan_dl = val_dl   # swap to train_dl / a Subset loader as needed

best_mse        = float("inf")
best_mse_sample = None   # (mse, inp_cpu, clear_cpu)

best_ssim        = -1.0
best_ssim_sample = None  # (ssim, inp_cpu, clear_cpu)

with torch.no_grad():
    for inp, clear in scan_dl:
        inp_d, clear_d = inp.to(device), clear.to(device)
        pred_opt, _, _ = model(inp_d)
        pred_opt = pred_opt.clamp(0, 1)

        for i in range(inp_d.shape[0]):
            p = pred_opt[i:i+1]
            t = clear_d[i:i+1]

            mse_i  = F.mse_loss(p, t).item()
            ssim_i = ssim_metric(p, t).item()

            if mse_i < best_mse:
                best_mse = mse_i
                best_mse_sample = (mse_i, inp[i:i+1].clone(), clear[i:i+1].clone())

            if ssim_i > best_ssim:
                best_ssim = ssim_i
                best_ssim_sample = (ssim_i, inp[i:i+1].clone(), clear[i:i+1].clone())

print(f"Best MSE:  {best_mse:.6f}")
print(f"Best SSIM: {best_ssim:.4f}")

## 19 · Inference Timing Benchmark


In [ ]:
import time
import torch
import torch.nn.functional as F
import numpy as np

# ---------------------------------------------------------------- #
# METRIC FUNCTIONS
# ---------------------------------------------------------------- #
def psnr_metric(pred, target, max_val=1.0):
    mse = F.mse_loss(pred, target)
    return 10 * torch.log10(max_val ** 2 / (mse + 1e-8))


def ssim_metric(pred, target, window_size=11):
    """Returns SSIM (not loss) in [0, 1]."""
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ch  = pred.shape[1]
    k   = torch.ones(ch, 1, window_size, window_size, device=pred.device) / (window_size ** 2)
    pad = window_size // 2

    mu1 = F.conv2d(pred,   k, padding=pad, groups=ch)
    mu2 = F.conv2d(target, k, padding=pad, groups=ch)
    mu1_sq, mu2_sq, mu12 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    s1  = F.conv2d(pred   ** 2, k, padding=pad, groups=ch) - mu1_sq
    s2  = F.conv2d(target ** 2, k, padding=pad, groups=ch) - mu2_sq
    s12 = F.conv2d(pred * target, k, padding=pad, groups=ch) - mu12

    ssim_map = ((2 * mu12 + C1) * (2 * s12 + C2)) / \
               ((mu1_sq + mu2_sq + C1) * (s1 + s2 + C2))
    return ssim_map.mean()


def sam_metric(pred, target, eps=1e-8):
    """Spectral Angle Mapper, averaged over the batch, in degrees.
    pred/target: [B, C, H, W] with C = spectral channels (e.g. RGB)."""
    p = pred.permute(0, 2, 3, 1).reshape(-1, pred.shape[1])      # [N, C]
    t = target.permute(0, 2, 3, 1).reshape(-1, target.shape[1])  # [N, C]
    dot = (p * t).sum(dim=1)
    norm_p = p.norm(dim=1)
    norm_t = t.norm(dim=1)
    cos_angle = dot / (norm_p * norm_t + eps)
    cos_angle = cos_angle.clamp(-1 + eps, 1 - eps)
    angle_rad = torch.acos(cos_angle)
    angle_deg = angle_rad * (180.0 / np.pi)
    return angle_deg.mean()


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ---------------------------------------------------------------- #
# RUN FULL EVAL OVER VAL SET  (swap val_dl -> train_dl if you want train-set numbers)
# ---------------------------------------------------------------- #
eval_dl = train_dl_small        # or: eval_dl = train_dl
device  = next(model.parameters()).device

total_params, trainable_params = count_parameters(model)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

model.eval()
psnr_sum, ssim_sum, sam_sum, n_batches = 0.0, 0.0, 0.0, 0
total_images = 0

torch.cuda.synchronize() if device.type == "cuda" else None
t_start = time.time()

with torch.no_grad():
    for inp, clear in eval_dl:
        inp, clear = inp.to(device), clear.to(device)
        pred_opt, _, _ = model(inp)
        pred_opt = pred_opt.clamp(0, 1)

        psnr_sum += psnr_metric(pred_opt, clear).item()
        ssim_sum += ssim_metric(pred_opt, clear).item()
        sam_sum  += sam_metric(pred_opt, clear).item()
        n_batches += 1
        total_images += inp.shape[0]

torch.cuda.synchronize() if device.type == "cuda" else None
t_end = time.time()

total_time = t_end - t_start
avg_time_per_image = total_time / max(total_images, 1)

final_psnr = psnr_sum / n_batches
final_ssim = ssim_sum / n_batches
final_sam  = sam_sum / n_batches

print(f"\n--- Final metrics over {total_images} images ({n_batches} batches) ---")
print(f"Final PSNR: {final_psnr:.3f} dB")
print(f"Final SSIM: {final_ssim:.4f}")
print(f"Final SAM:  {final_sam:.3f} degrees")
print(f"Total inference time: {total_time:.3f} s")
print(f"Avg time/image:       {avg_time_per_image*1000:.2f} ms")

## 20 · Random Training Sample Visualisation


In [ ]:
from torch.utils.data import Subset
import random

# pick N random samples from the training set
N = 50  # adjust as needed
indices = random.sample(range(len(train_ds)), N)
train_subset = Subset(train_ds, indices)

train_dl_small = DataLoader(train_subset, batch_size=4, shuffle=False,
                             num_workers=4, pin_memory=False)

## 21 · Full-Grid Panel Plot (Channels)


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------- #
# Channel layout (from SatDataDataset.__getitem__):
#   0:3   -> cloudy RGB
#   3:5   -> SAR (2ch)
#   5:8   -> temporal RGB
#   8:9   -> DEM
#   9:10  -> cloud_mask
#   10:11 -> dist_map
#   11:12 -> ndvi
# clear  -> ground-truth clear RGB  (separate tensor from dataloader)
# ---------------------------------------------------------------- #

device = next(model.parameters()).device
model.eval()

# ---- grab one sample (change index if you want a different tile) ----
inp, clear = next(iter(eval_dl))   # use val_dl / train_dl_small, whichever you've set
sample_idx = 0

inp_d   = inp.to(device)
clear_d = clear.to(device)

with torch.no_grad():
    pred_opt, pred_sar, feat = model(inp_d)

pred_opt = pred_opt.clamp(0, 1)
pred_sar = pred_sar.clamp(0, 1)

# ---- pull out single sample, move to CPU numpy ----
def to_np(t):
    return t[sample_idx].detach().cpu().numpy()

cloudy_rgb   = to_np(inp_d[:, 0:3]).transpose(1, 2, 0)
sar_in       = to_np(inp_d[:, 3:5])                       # [2,H,W]
temporal_rgb = to_np(inp_d[:, 5:8]).transpose(1, 2, 0)
dem          = to_np(inp_d[:, 8:9])[0]
cloud_mask   = to_np(inp_d[:, 9:10])[0]
dist_map     = to_np(inp_d[:, 10:11])[0]
ndvi         = to_np(inp_d[:, 11:12])[0]

clear_rgb    = to_np(clear_d).transpose(1, 2, 0)
pred_rgb     = to_np(pred_opt).transpose(1, 2, 0)
pred_sar_np  = to_np(pred_sar)                              # [2,H,W]

# per-channel SAR -> single-band displays (VV / VH or similar)
sar_in_vv, sar_in_vh   = sar_in[0], sar_in[1]
sar_pred_vv, sar_pred_vh = pred_sar_np[0], pred_sar_np[1]

# quick per-image metrics for this sample (uses functions from eval cell, if defined)
mse  = ((pred_rgb - clear_rgb) ** 2).mean()
psnr_val = 10 * np.log10(1.0 / (mse + 1e-8))

# ---------------------------------------------------------------- #
# PLOT EVERYTHING
# ---------------------------------------------------------------- #
fig, axes = plt.subplots(3, 4, figsize=(20, 15))

panels = [
    (cloudy_rgb.clip(0, 1),   "Cloudy Input (RGB)",      None),
    (clear_rgb.clip(0, 1),    "Ground Truth (Clear RGB)", None),
    (pred_rgb.clip(0, 1),     f"Predicted RGB (PSNR={psnr_val:.2f})", None),
    (temporal_rgb.clip(0, 1), "Temporal Reference (RGB)", None),

    (sar_in_vv,    "SAR Input - Ch1 (VV)",      "gray"),
    (sar_in_vh,    "SAR Input - Ch2 (VH)",      "gray"),
    (sar_pred_vv,  "SAR Predicted - Ch1 (VV)",  "gray"),
    (sar_pred_vh,  "SAR Predicted - Ch2 (VH)",  "gray"),

    (dem,          "DEM",                       "terrain"),
    (cloud_mask,   "Cloud Mask",                "gray"),
    (dist_map,     "Distance Map",              "viridis"),
    (ndvi,         "NDVI",                      "RdYlGn"),
]

for ax, (img, title, cmap) in zip(axes.flat, panels):
    if cmap is None:
        ax.imshow(img)
    else:
        ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

plt.suptitle("DGMR / MRNet -- Full Sample Visualization (all channels + predictions)",
             fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/full_sample_visualization.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved to /kaggle/working/full_sample_visualization.png")
print(f"Sample PSNR: {psnr_val:.3f} dB")

## 22 · Top-K Thick Cloud – Visual Export


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import heapq

device = next(model.parameters()).device
model.eval()

# ---------------------------------------------------------------- #
# STEP 1: scan dataset, keep only samples with enough cloud coverage,
# then keep TOP 5 highest-PSNR among those.
# ---------------------------------------------------------------- #
scan_dl = val_dl   # swap to train_dl_small / a Subset loader to limit scan time

TOP_K = 5
MIN_CLOUD_FRACTION = 0.60   # require at least 15% of pixels to be cloudy
                             # raise/lower this to control how "cloudy" the picks are

top_samples = []   # list of (psnr, cloud_fraction, inp_cpu, clear_cpu)

with torch.no_grad():
    for inp, clear in scan_dl:
        inp_d, clear_d = inp.to(device), clear.to(device)
        pred_opt, _, _ = model(inp_d)
        pred_opt = pred_opt.clamp(0, 1)

        for i in range(inp_d.shape[0]):
            cloud_mask_i    = inp_d[i, 9]               # [H, W], 1 = cloud
            cloud_fraction  = cloud_mask_i.mean().item()

            if cloud_fraction < MIN_CLOUD_FRACTION:
                continue   # skip near-clear tiles

            mse    = F.mse_loss(pred_opt[i], clear_d[i]).item()
            psnr_i = 10 * np.log10(1.0 / (mse + 1e-8))

            entry = (psnr_i, cloud_fraction, inp[i:i+1].clone(), clear[i:i+1].clone())

            if len(top_samples) < TOP_K:
                heapq.heappush(top_samples, entry)
            elif psnr_i > top_samples[0][0]:
                heapq.heapreplace(top_samples, entry)

top_samples.sort(key=lambda x: x[0], reverse=True)

print(f"Found {len(top_samples)} cloudy samples (>= {MIN_CLOUD_FRACTION*100:.0f}% cloud) "
      f"in top-{TOP_K} by PSNR:")
for rank, (p, cf, _, _) in enumerate(top_samples, 1):
    print(f"  #{rank}: PSNR={p:.3f} dB | cloud coverage={cf*100:.1f}%")

if len(top_samples) < TOP_K:
    print(f"\nNote: only found {len(top_samples)} samples meeting the cloud threshold "
          f"in this scan. Lower MIN_CLOUD_FRACTION or scan a larger/different dataloader "
          f"(e.g. full val_dl or train_dl) to get more.")

# ---------------------------------------------------------------- #
# STEP 2: generate + save full visualization for each selected sample
# ---------------------------------------------------------------- #
def to_np(t, idx=0):
    return t[idx].detach().cpu().numpy()

for rank, (psnr_recorded, cloud_fraction, best_inp, best_clear) in enumerate(top_samples, 1):
    inp_d   = best_inp.to(device)
    clear_d = best_clear.to(device)

    with torch.no_grad():
        pred_opt, pred_sar, feat = model(inp_d)
    pred_opt = pred_opt.clamp(0, 1)
    pred_sar = pred_sar.clamp(0, 1)

    cloudy_rgb   = to_np(inp_d[:, 0:3]).transpose(1, 2, 0)
    sar_in       = to_np(inp_d[:, 3:5])
    temporal_rgb = to_np(inp_d[:, 5:8]).transpose(1, 2, 0)
    dem          = to_np(inp_d[:, 8:9])[0]
    cloud_mask   = to_np(inp_d[:, 9:10])[0]
    dist_map     = to_np(inp_d[:, 10:11])[0]
    ndvi         = to_np(inp_d[:, 11:12])[0]

    clear_rgb    = to_np(clear_d).transpose(1, 2, 0)
    pred_rgb     = to_np(pred_opt).transpose(1, 2, 0)
    pred_sar_np  = to_np(pred_sar)

    sar_in_vv, sar_in_vh     = sar_in[0], sar_in[1]
    sar_pred_vv, sar_pred_vh = pred_sar_np[0], pred_sar_np[1]

    mse      = ((pred_rgb - clear_rgb) ** 2).mean()
    psnr_val = 10 * np.log10(1.0 / (mse + 1e-8))

    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    panels = [
        (cloudy_rgb.clip(0, 1),   f"Cloudy Input (RGB) -- {cloud_fraction*100:.1f}% cloud", None),
        (clear_rgb.clip(0, 1),    "Ground Truth (Clear RGB)", None),
        (pred_rgb.clip(0, 1),     f"Predicted RGB (PSNR={psnr_val:.2f})", None),
        (temporal_rgb.clip(0, 1), "Temporal Reference (RGB)", None),

        (sar_in_vv,    "SAR Input - Ch1 (VV)",      "gray"),
        (sar_in_vh,    "SAR Input - Ch2 (VH)",      "gray"),
        (sar_pred_vv,  "SAR Predicted - Ch1 (VV)",  "gray"),
        (sar_pred_vh,  "SAR Predicted - Ch2 (VH)",  "gray"),

        (dem,          "DEM",                       "terrain"),
        (cloud_mask,   "Cloud Mask",                "gray"),
        (dist_map,     "Distance Map",              "viridis"),
        (ndvi,         "NDVI",                      "RdYlGn"),
    ]

    for ax, (img, title, cmap) in zip(axes.flat, panels):
        ax.imshow(img, cmap=cmap) if cmap else ax.imshow(img)
        ax.set_title(title, fontsize=11)
        ax.axis("off")

    plt.suptitle(f"Rank #{rank} Cloudy Sample -- PSNR = {psnr_val:.2f} dB, "
                 f"Cloud Coverage = {cloud_fraction*100:.1f}%",
                 fontsize=15, fontweight="bold")
    plt.tight_layout()

    out_path = f"/kaggle/working/top{rank}_cloudy_psnr{psnr_val:.2f}.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

print("\nAll cloudy top-PSNR sample visualizations saved to /kaggle/working/")